Simulated Annealing Experiments

In [33]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
import logging

logging.basicConfig(level=logging.INFO, format='%(message)s')

In [35]:
from datetime import datetime

# Parameters
results_path = f'data/simulated_annealing/summary_{datetime.now().strftime("%Y-%m-%d_%H-%M-%S")}.npy'
brute_force_data_directory = './data/brute_force/first_run'

num_runs = 1000

# I think init and end temperatures should be chosen dependent on the delta of the energy,
# i.e. delta = new_loss - old_loss (in the exponent)
# larger delta -> larger temperature (and conversely smaller beta)
init_temp = 1000
end_temp = 1
max_temp_iterations = 80

In [36]:
import numpy as np

np.random.seed(4321) # set global seed for reproducibility

In [37]:
from pathlib import Path
import pandas as pd
import re

extract_candidate_dimension = lambda p: np.log2(int(re.search(string=str(p.name), pattern=r'_(\d*)_candidates').group(1)))
extract_load_factor = lambda p: float(re.search(string=str(p.name), pattern=r'_(\d*\.\d*)_load_factor').group(1))

data_files = [p for p in Path(brute_force_data_directory).glob("*.csv") if p.is_file()]
candidate_dims = [(extract_candidate_dimension(p), extract_load_factor(p), p) for p in data_files]

df = pd.DataFrame(data=candidate_dims, columns=['dim', 'load factor', 'path'])
df = df.groupby(['dim', 'load factor'])['path'].agg(list).reset_index()

In [38]:
problem_sizes = df['dim'].unique()
load_factors = df['load factor'].unique()
brute_force_data_paths = np.array(df['path'].to_list()).reshape(len(problem_sizes), len(load_factors), -1)

problem_sizes = problem_sizes

In [39]:
import pandas as pd
import numpy as np
brute_force_data = np.vectorize(pd.read_csv)(brute_force_data_paths)
loss_spaces = [np.array([d['loss'] for d in da]) for data in brute_force_data for da in data] # needs to be list because is inhomogeneous

In [40]:
from cl_optimizer import SimulatedAnnealing

def create_simulated_annealing_instances(x: np.ndarray):
    return np.array([[
        SimulatedAnnealing(
            lookup_table=data_path,
            bounds=int(x[0]),
        ) for _ in range(num_runs)
    ] for data_path in x[1:]])

sim_annss = np.swapaxes(np.array([np.apply_along_axis(func1d=create_simulated_annealing_instances, arr=np.column_stack((problem_sizes, brute_force_2d)), axis=1) for brute_force_2d in np.swapaxes(brute_force_data, 0, 1)]), 0, 1) # a sim_ann per problem size, instance and load factor

In [41]:
sim_annss.shape

(19, 10, 10, 1000)

In [42]:
temp_iterations = np.arange(10, max_temp_iterations + 10, 10)

In [43]:
def execute_in_parallel(problem_size: int, sas_per_problem_size: list):
    resultsss = []
    for j, sas_per_problem_instance in enumerate(sas_per_problem_size):
        resultss = []
        for k, sas_per_load_factor in enumerate(sas_per_problem_instance):
            results = []
            for l, sim_ann in enumerate(sas_per_load_factor):
                print(f"Problem size: {problem_size + 1}/{int(problem_sizes[-1])} - instance: {j + 1}/{len(sas_per_problem_instance)} - load factor: {k + 1}/{len(load_factors)} - run: {l + 1}/{len(sas_per_load_factor)}")
                results.append(get_execute_sim_ann(init_temp, end_temp)(sim_ann, temp_iterations))

            resultss.append(results)
        resultsss.append(resultss)

    return resultsss

In [44]:
from joblib import Parallel, delayed
from parallelization import get_execute_sim_ann

# No logs will be printed below the cell as jupyter and joblib seem to be not very compatible regarding logging - use papermill instead if you need observability
resultssss = Parallel(n_jobs=len(problem_sizes), verbose=50)(delayed(execute_in_parallel)(i, sas_per_problem_size) for i, sas_per_problem_size in enumerate(sim_annss))


[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.


KeyboardInterrupt: 

In [ ]:
result_tensor = np.stack(resultssss, axis=0).squeeze()

In [ ]:
indices = np.argsort(result_tensor[..., 0], axis=3) # sort by temp. iterations (parallelization does not guarantee order) along runs (axis=3)
indices_expanded = np.expand_dims(indices, axis=-1)
result_tensor = np.take_along_axis(result_tensor, indices_expanded, axis=3)

In [ ]:
result_tensor.shape # should be a 6-tuple: (n_problem_sizes, n_instances_per_problem_size, n_load_factors_per_instance, n_runs_per_load_factor, n_temp_iter_steps, 2)

In [ ]:
np.save(results_path, result_tensor)